# Reading from the Bronze Table

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import *
from pyspark.sql.functions import trim,col

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

In [0]:
df.display()


# Data Transformation

In [0]:
# trim the string
# normalization for the marital status and gender
#column name nees=ds to be changed

for field in  df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.display()

In [0]:
# Normalization
df = (
    df.withColumn(
        "cst_marital_status", f.when(col("cst_marital_status") == "M", "MARRIED")
        .otherwise("SINGLE")
    )
    .withColumn(
        "cst_gndr", f.when(col("cst_gndr") == "M", "MALE")
        .otherwise("FEMALE")
    )
)

In [0]:
# rename
Rename_map = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date",
}



In [0]:
#rename column
for old_name, new_name in Rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

#

# Write to Silver Table

In [0]:
#write to delta
df.write.mode("overwrite").saveAsTable("silver.customer_info")

In [0]:
%sql
select * from silver.customer_info